# Universal Multimodal Sentiment Analysis Pipeline 
This notebook provides an interactive playground for exploring the refactored multimodal codebase. We analyze datasets, process input streams (Text + Vision + Audio) and evaluate state-of-the-art sentiment predictions.

In [ ]:
import os
import sys

miniconda_path = f"{os.environ.get('HOME', '')}/miniconda/bin"
os.environ["PATH"] = f"{miniconda_path}:" + os.environ.get("PATH", "")

print(f"Conda PATH securely bound to: {miniconda_path}")

In [ ]:
import subprocess
import os

env_name = "multimodal_env"
env_file = "../environment.yml"

print(f"Checking for Conda environment {env_name}...")
try:
    result = subprocess.run(["conda", "env", "list"], capture_output=True, text=True, check=True)
    if env_name in result.stdout:
        print(f"Environment {env_name} found. Updating from {env_file}...")
        subprocess.run(["conda", "env", "update", "-f", env_file, "--prune"], check=True)
    else:
        print(f"Environment {env_name} not found. Creating from {env_file}...")
        subprocess.run(["conda", "env", "create", "-f", env_file], check=True)
    print("Multimodal Conda Environment check complete.")
except subprocess.CalledProcessError as e:
    print(f"Error executing Anaconda operations: {e}")
except FileNotFoundError:
    print("Conda binary not found in PATH.")

In [ ]:
import os
import sys
# Ensure our src folder is loaded
sys.path.append(os.path.abspath('..'))

from src.config import DATA_DIR
from src.data.ingestion import download_msctd
from src.data.preprocess import extract_faces, sent_preprocess

## 1. Data Ingestion & Preprocessing
Download the text and image components manually or via the API.

In [ ]:
# Caution: This downloads several gigabytes of data
# download_msctd()
print('Data paths configured to: ', DATA_DIR)

## 2. Text Preprocessing
Testing advanced text normalization and cleaning leveraging NLP heuristics.

In [ ]:
sample_text = "I am extremely happy today! 😄 1st time visiting NYC."
cleaned = sent_preprocess(sample_text, lemmatize=True, remove_emojies=True)
print(f"Original: {sample_text}\nCleaned: {cleaned}")

## 3. Loading the Multimodal Fusion Model
We load the fusion architecture initialized with robust modern backbones: `RoBERTa` for NLP, `ViT` for images, and `Wav2Vec2` for audio.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoFeatureExtractor
from src.models.multimodal import MultimodalFusionNet
from src.config import TEXT_MODEL_NAME, VISION_BACKBONE_NAME

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultimodalFusionNet(use_audio=True).to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
feature_extractor = AutoFeatureExtractor.from_pretrained(VISION_BACKBONE_NAME)
print("Model instantiated properly on", device)

## 4. Inference Test
We pass an artificial tensor test to simulate live performance.

In [ ]:
with torch.no_grad():
    sample_txt = tokenizer(["This looks bad."], return_tensors="pt").to(device)
    sample_img = torch.randn(1, 3, 224, 224).to(device)
    sample_aud = torch.randn(1, 16000).to(device) # 1 sec dummy audio
    
    logits = model(sample_txt['input_ids'], sample_txt['attention_mask'], sample_img, sample_aud)
    print("Logits:", logits)
    probs = torch.softmax(logits, dim=1)
    print("Probabilities [Negative, Neutral, Positive]:", probs)